# MOD-08 · NB-01 — Reproducible Research Foundations
### Health Analytics with Python · Module 08: Reproducible Research & Deployment
---
**Learning objectives**
- Understand reproducibility threats in health analytics
- Build canonical project structure with dependency management
- Implement data provenance with SHA-256 checksums and MANIFEST.json
- Build reproducible pipelines with structured logging and decorators
- Write Makefiles for pipeline orchestration
- Conduct a reproducibility audit

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `hashlib`, `logging`, `json`, `pathlib`

## 1. Reproducibility Threats

| Threat | Example | Prevention |
|--------|---------|------------|
| Random seeds | ML results change | `np.random.seed(42)` everywhere |
| Package versions | sklearn API breaks pipelines | Pin versions in requirements.txt |
| Data drift | Source updated silently | Checksums + MANIFEST.json |
| Hidden notebook state | Cells run out of order | Restart & Run All + CI |
| OS differences | Float rounding varies | Docker |
| Undocumented transforms | 'I normalised this manually' | Code-only pipelines |

## 2. Project Setup & Dependencies

In [ ]:
import os, json, hashlib, logging, sys, time
from pathlib import Path
from datetime import datetime
from functools import wraps
import numpy as np, pandas as pd
import os; os.makedirs("/tmp/mod08", exist_ok=True)

PROJECT_ROOT = Path("/tmp/mod08/demo_project")
for d in ["data/raw","data/processed","src/data","src/models","models","reports/figures","tests","notebooks"]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)
print("Project structure created at:", PROJECT_ROOT)


In [ ]:
REQUIREMENTS = (
    "# Health Analytics Project Requirements\n"
    "numpy>=1.24,<2.0\n"
    "pandas>=2.0,<3.0\n"
    "scikit-learn>=1.3,<2.0\n"
    "scipy>=1.11,<2.0\n"
    "matplotlib>=3.7,<4.0\n"
    "seaborn>=0.13,<0.14\n"
    "statsmodels>=0.14,<0.15\n"
    "xgboost>=2.0,<3.0\n"
    "lightgbm>=4.0,<5.0\n"
    "shap>=0.44,<0.45\n"
    "streamlit>=1.28,<2.0\n"
    "mlflow>=2.8,<3.0\n"
    "fastapi>=0.104,<0.105\n"
    "uvicorn>=0.24,<0.25\n"
    "pydantic>=2.0,<3.0\n"
    "pytest>=7.4,<8.0\n"
)
(PROJECT_ROOT / "requirements.txt").write_text(REQUIREMENTS)
print("requirements.txt written with pinned version ranges:")
print(REQUIREMENTS)


## 3. Data Provenance & Checksums

In [ ]:
def compute_checksum(filepath, algorithm="sha256"):
    h = hashlib.new(algorithm)
    p = Path(filepath)
    if p.is_file():
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(8192), b""):
                h.update(chunk)
        return h.hexdigest()
    return "file_not_found"

def create_manifest(data_info, output_path):
    manifest = {
        "created_at"    : datetime.now().isoformat(),
        "python_version": sys.version.split()[0],
        "files"         : {}
    }
    for name, info in data_info.items():
        manifest["files"][name] = {
            "path"        : info.get("path",""),
            "source"      : info.get("source",""),
            "download_date": info.get("download_date", datetime.now().date().isoformat()),
            "sha256"      : compute_checksum(info.get("path","")),
            "description" : info.get("description",""),
            "n_rows"      : info.get("n_rows"),
            "n_cols"      : info.get("n_cols"),
        }
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    Path(output_path).write_text(json.dumps(manifest, indent=2, default=str))
    return manifest

# Generate synthetic dataset
np.random.seed(42); N = 2000
df_raw = pd.DataFrame({
    "patient_id" : [f"PT-{i:05d}" for i in range(N)],
    "age"        : np.random.normal(60,15,N).clip(18,90).astype(int),
    "readmitted" : np.random.binomial(1, 0.18, N),
    "los_days"   : np.random.gamma(2.5,1.8,N).clip(1,30).astype(int),
    "diabetes"   : np.random.binomial(1, 0.35, N),
    "hf"         : np.random.binomial(1, 0.22, N),
})
raw_path = PROJECT_ROOT / "data" / "raw" / "readmission_cohort.csv"
df_raw.to_csv(raw_path, index=False)

manifest = create_manifest(
    {"readmission_cohort": {
        "path": str(raw_path), "source": "Synthetic (MIMIC-III inspired), seed=42",
        "description": "30-day readmission cohort", "n_rows": len(df_raw), "n_cols": len(df_raw.columns),
    }},
    output_path=str(PROJECT_ROOT / "data" / "MANIFEST.json")
)
sha = manifest["files"]["readmission_cohort"]["sha256"]
print(f"Dataset: {len(df_raw)} rows | SHA-256: {sha[:12]}...{sha[-8:]}")
print("MANIFEST.json written.")


## 4. Reproducible Pipeline with Logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    handlers=[logging.FileHandler(str(PROJECT_ROOT/"pipeline.log")), logging.StreamHandler()],
)
logger = logging.getLogger("health_pipeline")

def log_step(name):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            logger.info(f"START  {name}")
            t0 = time.time()
            try:
                result = fn(*args, **kwargs)
                logger.info(f"OK     {name} | {time.time()-t0:.2f}s")
                return result
            except Exception as e:
                logger.error(f"FAIL   {name} | {e}"); raise
        return wrapper
    return decorator

@log_step("load_raw_data")
def load_raw(path):
    df = pd.read_csv(path)
    logger.info(f"  Loaded {len(df)}r x {len(df.columns)}c")
    return df

@log_step("validate_schema")
def validate(df):
    checks = {
        "patient_id_unique": df.patient_id.nunique() == len(df),
        "age_18_90"        : df.age.between(18,90).all(),
        "readmitted_binary" : df.readmitted.isin([0,1]).all(),
        "los_positive"      : (df.los_days > 0).all(),
    }
    for check, ok in checks.items():
        if not ok: raise ValueError(f"Validation FAILED: {check}")
        logger.info(f"  PASS: {check}")
    return df

@log_step("feature_engineering")
def engineer_features(df):
    df = df.copy()
    df["los_gt7"]  = (df.los_days > 7).astype(int)
    df["age_std"]  = (df.age - df.age.mean()) / df.age.std()
    df["n_comorb"] = df.diabetes + df.hf
    logger.info(f"  Added 3 features | Output: {df.shape}")
    return df

df_loaded   = load_raw(raw_path)
df_valid    = validate(df_loaded)
df_features = engineer_features(df_valid)
proc_path   = PROJECT_ROOT / "data" / "processed" / "features.parquet"
df_features.to_parquet(proc_path, index=False)
print(f"Pipeline complete | Processed: {df_features.shape}")


## 5. Makefile Pipeline Orchestration

In [ ]:
MAKEFILE_CONTENT = (
    "# Health Analytics Pipeline Makefile\n"
    ".PHONY: all clean data features train evaluate report test\n\n"
    "all: data features train evaluate report\n"
    "\t@echo '[OK] Full pipeline complete'\n\n"
    "data:\n"
    "\tpython3 src/data/download.py --output data/raw/\n"
    "\tpython3 src/data/validate.py --manifest data/MANIFEST.json\n\n"
    "features: data\n"
    "\tpython3 src/data/features.py --input data/raw/ --output data/processed/\n\n"
    "train: features\n"
    "\tpython3 src/models/train.py --seed 42 --output models/readmission_v1.pkl\n\n"
    "evaluate: train\n"
    "\tpython3 src/models/evaluate.py --output reports/metrics.json\n\n"
    "report: evaluate\n"
    "\tquarto render notebooks/readmission_report.qmd --to html\n\n"
    "test:\n"
    "\tpytest tests/ -v --tb=short\n\n"
    "clean:\n"
    "\trm -rf data/processed/ models/ reports/figures/\n"
    "\t@echo 'Cleaned. Raw data preserved.'\n"
)
(PROJECT_ROOT / "Makefile").write_text(MAKEFILE_CONTENT)
print("Makefile written:")
print(MAKEFILE_CONTENT)


## 6. Reproducibility Audit

In [ ]:
def reproducibility_audit(project_dir):
    root = Path(project_dir); issues = []; score = 100

    req = root / "requirements.txt"
    if not req.exists():
        issues.append("MISSING requirements.txt"); score -= 20
    else:
        unpinned = [l.strip() for l in req.read_text().split("\n")
                    if l.strip() and not l.startswith("#")
                    and not any(op in l for op in ["==",">=","<=","~=","!="])]
        if unpinned:
            issues.append(f"Unpinned packages: {unpinned}"); score -= 10

    if not (root/"data"/"MANIFEST.json").exists():
        issues.append("MISSING data/MANIFEST.json"); score -= 20

    gi = root / ".gitignore"
    if gi.exists():
        content = gi.read_text()
        for p in [".env","data/raw/"]:
            if p not in content:
                issues.append(f".gitignore missing: {p}"); score -= 5
    else:
        issues.append("MISSING .gitignore"); score -= 15

    if not (root/"pipeline.log").exists():
        issues.append("No pipeline.log found"); score -= 5

    return {"score": max(0,score), "issues": issues}

(PROJECT_ROOT / ".gitignore").write_text(".env\ndata/raw/\nmodels/\n__pycache__/\n*.pyc\n")

result = reproducibility_audit(str(PROJECT_ROOT))
print(f"Reproducibility Score: {result['score']}/100")
for issue in result["issues"]:
    print(f"  [WARN] {issue}")
if not result["issues"]:
    print("  All checks passed!")


## 7. Knowledge Check
1. A colleague says 'I ran all cells top-to-bottom — it is reproducible.' Why might this still fail?
2. Your requirements.txt pins `scikit-learn>=1.3,<2.0`. Six months later predictions differ. What happened?
3. How does SHA-256 checksumming protect against silent data warehouse updates?
4. The Makefile re-runs `train` if `features.parquet` is newer. A bug in `train.py` is fixed. Does `make train` re-run?
5. Write a conftest.py fixture that verifies MANIFEST.json checksums before any test runs.

In [ ]:
# Exercise 5 — pytest conftest fixture for checksum validation
CONFTEST = (
    "# tests/conftest.py\n"
    "import pytest, json, hashlib\n"
    "from pathlib import Path\n\n"
    "@pytest.fixture(scope='session', autouse=True)\n"
    "def verify_data_integrity():\n"
    "    manifest_path = Path('data/MANIFEST.json')\n"
    "    if not manifest_path.exists():\n"
    "        pytest.skip('No MANIFEST.json')\n"
    "    manifest = json.loads(manifest_path.read_text())\n"
    "    for name, info in manifest['files'].items():\n"
    "        p = Path(info['path'])\n"
    "        if not p.exists(): pytest.fail(f'Missing: {p}')\n"
    "        h = hashlib.sha256()\n"
    "        with open(p,'rb') as f:\n"
    "            for chunk in iter(lambda:f.read(8192),b''): h.update(chunk)\n"
    "        if info.get('sha256') and h.hexdigest() != info['sha256']:\n"
    "            pytest.fail(f'Checksum FAILED for {name} -- data may have changed!')\n"
    "    print(f'[OK] {len(manifest.files)} files passed integrity check')\n"
)
print(CONFTEST)


---
**Next → NB-02: Quarto Documents for Health Research Reports**